# 20×20 PT + rotation：固定预算运行

先将整个文件夹放在 `lattice-gas` 下，与原 notebook 同级，然后使用当前 Python kernel。
本 notebook 不依赖原 notebook 的内存变量，也不会覆盖旧结果。

**执行顺序：小规模检查 → 创建并冻结目录 → 正式运行。**
旧 reference 的 NOT READY 状态不变。下面的正式运行结束后仍需单独做统计评估。


In [1]:
from pathlib import Path
import json
import pt_rotation_runner as runner
from test_runner import run_checks

check_report = run_checks()
print(json.dumps(check_report, indent=2))


{
  "status": "IMPLEMENTATION_CHECKS_PASSED",
  "checks": [
    "20x20 initialization, generic FFT normalization, rotation invariants",
    "9072 exhaustive 3x3 N=4 energy-difference checks",
    "Continuous / checkpoint pause / mid-segment interrupt: 420 array-and-RNG comparisons identical",
    "Completed campaign rerun does not alter trajectories",
    "Concurrent lock, protocol modification, and missing segment rejected"
  ],
  "environment": {
    "python": "3.14.2",
    "numpy": "2.4.1",
    "platform": "Windows-11-10.0.26200-SP0"
  },
  "source_sha256": {
    "pt_rotation_runner.py": "2f281d4f012aef28a0ffecbaf286d974d987cc4acddb45a9e19655fbef8fe958",
    "pt_rotation_dependencies.py": "e28756f4811cce9da2d4997da34f48087b3a7e01c1c548e119c54fe1144c86e7"
  },
  "scope": "Small implementation tests only; no 20x20 production trajectory or equilibrium assessment."
}


## 创建新 campaign（只做一次，不运行 MC）

预算：9 个独立运行 × 50,000 cycles；8 个温度槽。
初态为 random / stripe / compact droplet，每类 3 个 seed。
前 10,000 cycles 不纳入统计；随后测量 40,000 cycles。
每 5,000 cycles 保存 checkpoint，构型与形状每 10 cycles 保存。

完整标准见 `protocol_preview.json` 和 `README.md`。
稀有形状概率保持探索性评估；q 不作为形状混合通过标准。

**请保留打印的完整目录。重启 kernel 后要恢复已有目录，不要重新创建 campaign。**


In [2]:
assert check_report["status"] == "IMPLEMENTATION_CHECKS_PASSED"
if "campaign_dir" not in globals():
    campaign_dir = runner.create_campaign(Path.cwd().parent / "results")
else:
    runner.read_manifest(campaign_dir)
    print("Reusing:", campaign_dir)
print("RESUME DIRECTORY:", campaign_dir)


Created: C:\Users\user\Documents\GitHub\active-matter-reproductions\lattice-gas\results\pt_rotation_20x20_reference_v1_20261103_220323_345300
{
  "runs": 9,
  "cycles_per_run": 50000,
  "within_slot_proposals": 1440000000,
  "swap_attempts": 3150000,
  "rotation_calls_including_identity": 3600000,
  "segments": 90,
  "post_exclusion_snapshots_per_run": 4000
}
No simulation started. Keep this exact path for resume.
RESUME DIRECTORY: C:\Users\user\Documents\GitHub\active-matter-reproductions\lattice-gas\results\pt_rotation_20x20_reference_v1_20261103_220323_345300


## 正式运行：运行下一格才开始

14.4 亿次槽内提议。此前旧 PT 预算的基础耗时约 84 分钟的外推只供参考；
本轮增加旋转、形状计算和保存，实际可能更久，以上并非运行时间承诺。

每 1,000 cycles 打印进度，每 5,000 cycles 打印 `Committed`。
需要暂停时可用中断按钮；最多重算尚未保存的一个 segment。
同一 kernel 中再次运行下格会自动恢复；已全部完成则不会增加模拟。


In [3]:
execution_report = runner.run_campaign(campaign_dir)


random  seed=1742 cycle=1000/50000 cold U=-356.0
random  seed=1742 cycle=2000/50000 cold U=-367.0
random  seed=1742 cycle=3000/50000 cold U=-357.0
random  seed=1742 cycle=4000/50000 cold U=-363.0
random  seed=1742 cycle=5000/50000 cold U=-352.0
Committed: segment_000000_005000.npz
random  seed=1742 cycle=6000/50000 cold U=-362.0
random  seed=1742 cycle=7000/50000 cold U=-366.0
random  seed=1742 cycle=8000/50000 cold U=-360.0
random  seed=1742 cycle=9000/50000 cold U=-360.0
random  seed=1742 cycle=10000/50000 cold U=-352.0
Committed: segment_005000_010000.npz
random  seed=1742 cycle=11000/50000 cold U=-361.0
random  seed=1742 cycle=12000/50000 cold U=-367.0
random  seed=1742 cycle=13000/50000 cold U=-354.0
random  seed=1742 cycle=14000/50000 cold U=-360.0
random  seed=1742 cycle=15000/50000 cold U=-362.0
Committed: segment_010000_015000.npz
random  seed=1742 cycle=16000/50000 cold U=-370.0
random  seed=1742 cycle=17000/50000 cold U=-361.0
random  seed=1742 cycle=18000/50000 cold U=-362.

## 重启 kernel 后的恢复方式

先运行下面这段代码（将路径替换为上面实际打印的目录）。
不要再次运行“创建新 campaign”格。

```python
from pathlib import Path
import pt_rotation_runner as runner
campaign_dir = Path(r"C:\Users\user\Documents\GitHub\active-matter-reproductions\lattice-gas\results\这里替换成实际目录名")
execution_report = runner.run_campaign(campaign_dir)
```

如果电脑关机或 kernel 被强制结束，可能留下 `RUNNING.lock`。
确认没有其他进程运行该 campaign 后，仅删除这个锁文件再恢复，保留所有数据段。
普通中断会自动清理锁。

结束状态应为 `FIXED_BUDGET_EXECUTION_COMPLETE_ASSESSMENT_PENDING`。
保存全部 NPZ。可先提供 `execution_summary.json` 和 `protocol.json`，随后做只读统计评估。
**运行完成本身不意味着 equilibrium reference 已建立。**
